# VisTacFusion — Real-Only Evaluation

Load a trained sim+real checkpoint and evaluate on **real data only**.
Only real data is needed — no sim data extraction.

**Prerequisites on Google Drive:**
- `MyDrive/HDR_Lab/real_data/real_data.zip` — real data
- `MyDrive/HDR_Lab/VisTacFusion/meshes.tar` — mesh .obj files
- `MyDrive/HDR_Lab/DINOv3_DPT/dinov3_vitl16_pretrain_lvd1689m.pth` — encoder weights
- `MyDrive/HDR_Lab/VisTacFusion/outputs/best_depth.pt` (and optionally `best_pose.pt`) — trained checkpoint

## 1. Setup: Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo
!git clone https://github.com/cynthiahuang1004/VisTacFusion.git /content/VisTacFusion
%cd /content/VisTacFusion
!git checkout VisTacFusion-v2

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install trimesh scipy -q

## 2. Data: Extract Real Data Only

In [ ]:
import os, glob, time

# ---- Drive paths ----
DRIVE_REAL_ZIP  = '/content/drive/MyDrive/HDR_Lab/real_data/real_data.zip'
DRIVE_MESHES    = '/content/drive/MyDrive/HDR_Lab/VisTacFusion/meshes.tar'
DRIVE_WEIGHTS   = '/content/drive/MyDrive/HDR_Lab/DINOv3_DPT/dinov3_vitl16_pretrain_lvd1689m.pth'

# ---- Checkpoint paths (update to match your training output) ----
DRIVE_OUTPUT    = '/content/drive/MyDrive/HDR_Lab/VisTacFusion/outputs'
DEPTH_CKPT_NAME = 'best_depth.pt'
POSE_CKPT_NAME  = 'best_pose.pt'

# ---- Local paths ----
LOCAL_REAL      = '/content/data/real_data'
LOCAL_MESHES    = '/content/data/meshes'
LOCAL_WEIGHTS   = '/content/VisTacFusion/weights/dinov3_vitl16_pretrain_lvd1689m.pth'

os.makedirs(os.path.dirname(LOCAL_WEIGHTS), exist_ok=True)

In [ ]:
# Extract meshes
if not os.path.exists(LOCAL_MESHES):
    os.makedirs('/content/data', exist_ok=True)
    print('Extracting meshes...')
    !tar xf "{DRIVE_MESHES}" -C /content/data/
    print(f'  -> {len(os.listdir(LOCAL_MESHES))} files')
else:
    print('Meshes already extracted')

In [ ]:
# Extract real data
if os.path.exists(LOCAL_REAL) and os.listdir(LOCAL_REAL):
    print(f'Real data already extracted -> {LOCAL_REAL}')
else:
    print(f'Extracting real data from {DRIVE_REAL_ZIP}...')
    t0 = time.time()
    !unzip -q "{DRIVE_REAL_ZIP}" -d /content/data/
    elapsed = time.time() - t0
    print(f'  done ({elapsed:.0f}s)')

    # Handle nested directory
    nested = os.path.join(LOCAL_REAL, 'real_data')
    if os.path.isdir(nested) and not any(
        d.startswith('pattern') or d.startswith('session')
        for d in os.listdir(LOCAL_REAL) if d != 'real_data'
    ):
        import shutil
        print('  fixing nested real_data/ directory...')
        for item in os.listdir(nested):
            shutil.move(os.path.join(nested, item), os.path.join(LOCAL_REAL, item))
        os.rmdir(nested)

real_objects = sorted(d for d in os.listdir(LOCAL_REAL)
                      if os.path.isdir(os.path.join(LOCAL_REAL, d)))
total_real = sum(
    len(glob.glob(os.path.join(LOCAL_REAL, o, '**/samples/*.png'), recursive=True))
    for o in real_objects
)
print(f'{len(real_objects)} objects, {total_real} samples')
print(f'Objects: {real_objects}')

In [ ]:
# Copy DINOv3 weights
if not os.path.exists(LOCAL_WEIGHTS):
    print('Copying DINOv3 weights...')
    !cp "{DRIVE_WEIGHTS}" "{LOCAL_WEIGHTS}"
    print(f'  -> {os.path.getsize(LOCAL_WEIGHTS) / 1e9:.1f} GB')
else:
    print('Weights already copied')

## 3. Configure & Build Real-Only Val Dataset

In [ ]:
import yaml

# --- data.yaml overrides ---
with open('configs/data.yaml') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['real']['root'] = LOCAL_REAL
data_cfg['real']['mesh_dir'] = LOCAL_MESHES
data_cfg['real']['rgb_subdir'] = 'rgb'
data_cfg['loader']['num_workers'] = 4
data_cfg['loader']['persistent_workers'] = False

with open('configs/data.yaml', 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

# --- model.yaml: object embedding on ---
with open('configs/model.yaml') as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokens']['object_embedding'] = True

with open('configs/model.yaml', 'w') as f:
    yaml.dump(model_cfg, f, default_flow_style=False, sort_keys=False)

print('Configs updated.')
print(f"  real.root: {data_cfg['real']['root']}")
print(f"  object_embedding: {model_cfg['tokens']['object_embedding']}")

In [ ]:
from vistacfusion.utils.config import merge_configs
from vistacfusion.data.dataset import SimVisuoTactileDataset

cfg = merge_configs('configs/model.yaml', 'configs/train.yaml', 'configs/data.yaml')

# Build real val dataset directly (no sim data needed)
val_ds = SimVisuoTactileDataset(
    cfg, cfg.image_size, augment=False, split='val', data_section='real')
print(f'Val (real-only): {len(val_ds)} samples')

# Also build real train for reference
real_train = SimVisuoTactileDataset(
    cfg, cfg.image_size, augment=False, split='train', data_section='real')
print(f'Real train: {len(real_train)} samples (for reference)')

## 4. Load Checkpoint

In [ ]:
import torch
from vistacfusion.models.model import build_model
from vistacfusion.engine.train import load_checkpoint

device = torch.device('cuda')

DEPTH_CKPT = os.path.join(DRIVE_OUTPUT, DEPTH_CKPT_NAME)
POSE_CKPT  = os.path.join(DRIVE_OUTPUT, POSE_CKPT_NAME)

# Load depth model
depth_model = build_model(cfg).to(device)
load_checkpoint(DEPTH_CKPT, depth_model, device=device)
depth_model.eval()
d_epoch = torch.load(DEPTH_CKPT, map_location='cpu', weights_only=False).get('epoch', '?')
print(f'Depth model: epoch {d_epoch}')

# Load pose model (optional)
pose_model = None
if os.path.exists(POSE_CKPT):
    pose_model = build_model(cfg).to(device)
    load_checkpoint(POSE_CKPT, pose_model, device=device)
    pose_model.eval()
    p_epoch = torch.load(POSE_CKPT, map_location='cpu', weights_only=False).get('epoch', '?')
    print(f'Pose model:  epoch {p_epoch}')
    ckpt_info = f'depth={DEPTH_CKPT_NAME} (epoch {d_epoch}), pose={POSE_CKPT_NAME} (epoch {p_epoch})'
else:
    ckpt_info = f'{DEPTH_CKPT_NAME} (epoch {d_epoch})'
    print('No separate pose checkpoint, using depth model for all heads')

## 5. Quantitative Eval (Real Val Set)

Runs all 3 configs (both, tactile, rgb) on real val set with GT comparison.

In [ ]:
import math
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from vistacfusion.engine.eval import precompute_encoder_cache, _slice_cache
from vistacfusion.engine.inference import CONFIGS_3, visualize_three_configs

EVAL_DIR = os.path.join(DRIVE_OUTPUT, 'eval_real_only')
vis_dir = os.path.join(EVAL_DIR, 'real_val_vis')
os.makedirs(vis_dir, exist_ok=True)

batch_size = 32
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                        num_workers=4, pin_memory=True)

# Pre-compute encoder cache
eval_model = pose_model if pose_model is not None else depth_model
print('Pre-computing encoder cache for real val set...')
encoder_cache = precompute_encoder_cache(eval_model, val_loader, device)

# Group samples by object for per-object visualization
obj_vis_idx = {}
for i, (unit, _sample_idx) in enumerate(val_ds.samples):
    obj_name = os.path.basename(os.path.dirname(os.path.dirname(unit)))
    if obj_name not in obj_vis_idx:
        obj_vis_idx[obj_name] = i
vis_indices = set(obj_vis_idx.values())

# Batched evaluation
metrics = {cfg_name: {
    'depth_mse': 0.0, 'normal_mse': 0.0, 'normal_mse_norm': 0.0,
    'depth_mae': [], 'depth_rmse': [], 'depth_d1': [],
    'normal_ang_mean': 0.0, 'normal_ang_median': 0.0,
    'pose_rot_l1': 0.0, 'pose_rot_deg': 0.0, 'pose_trans_l1': 0.0, 'n': 0,
} for cfg_name in CONFIGS_3}

vis_results = {}
mean_np = np.array([123.675, 116.28, 103.53])
std_np = np.array([58.395, 57.12, 57.375])

print(f'Evaluating {len(val_ds)} real val samples (3 configs, batched)...')
sample_idx = 0
for batch in val_loader:
    bs = batch['rgb'].shape[0]
    batch_dev = {k: (v.to(device) if torch.is_tensor(v) else v)
                 for k, v in batch.items()}
    batch_enc = _slice_cache(encoder_cache, sample_idx, sample_idx + bs, device)

    for cfg_name in CONFIGS_3:
        with torch.no_grad():
            out = depth_model(batch_dev['rgb'], batch_dev['tactile'],
                              config=cfg_name, encoder_cache=batch_enc)
            if pose_model is not None:
                pose_out = pose_model(batch_dev['rgb'], batch_dev['tactile'],
                                     config=cfg_name, encoder_cache=batch_enc)
                out['se2'] = pose_out.get('se2', pose_out.get('trans'))

        m = metrics[cfg_name]
        m['depth_mse'] += F.mse_loss(out['depth'], batch_dev['depth']).item() * bs
        m['normal_mse'] += F.mse_loss(out['normal'], batch_dev['normal']).item() * bs

        pred_normal = out['normal'].cpu()
        gt_normal = batch['normal']
        p_n = F.normalize(pred_normal.float(), dim=1)
        g_n = F.normalize(gt_normal.float(), dim=1)
        m['normal_mse_norm'] += F.mse_loss(p_n, g_n).item() * bs
        cos_sim = (p_n * g_n).sum(dim=1).clamp(-1, 1)
        angles = torch.acos(cos_sim) * (180.0 / math.pi)
        m['normal_ang_mean'] += angles.mean().item() * bs
        m['normal_ang_median'] += angles.median().item() * bs

        if 'se2' in out:
            se2 = out['se2'].float().cpu()
            gt_pose = batch['pose']
            cos_p, sin_p = se2[:, 0], se2[:, 1]
            cos_g, sin_g = gt_pose[:, 0], gt_pose[:, 1]
            theta_pred = torch.atan2(sin_p, cos_p)
            theta_gt = torch.atan2(sin_g, cos_g)
            rot_err = (theta_pred - theta_gt).abs()
            rot_err = torch.min(rot_err, 2 * math.pi - rot_err)
            m['pose_rot_l1'] += ((cos_p - cos_g).abs() + (sin_p - sin_g).abs()).mean().item() * bs
            m['pose_rot_deg'] += (rot_err * 180.0 / math.pi).mean().item() * bs
            m['pose_trans_l1'] += F.l1_loss(se2[:, 2:], gt_pose[:, 2:]).item() * bs

        pred_depth = out['depth'].cpu()
        for j in range(bs):
            pd = pred_depth[j, 0].numpy()
            gd = batch['depth'][j, 0].numpy()
            mask = gd > 0
            if mask.sum() > 0:
                p, g = pd[mask], gd[mask]
                abs_diff = np.abs(p - g)
                m['depth_mae'].append(float(abs_diff.mean()))
                m['depth_rmse'].append(float((abs_diff ** 2).mean()) ** 0.5)
                ratio = np.maximum(p / np.clip(g, 1e-6, None),
                                   g / np.clip(p, 1e-6, None))
                m['depth_d1'].append(float((ratio < 1.25).mean()) * 100)

        for j in range(bs):
            global_idx = sample_idx + j
            if global_idx in vis_indices:
                d = out['depth'][j, 0].cpu().numpy()
                n = out['normal'][j].cpu().permute(1, 2, 0).numpy()
                n = n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)
                if 'se2' in out:
                    p = out['se2'][j].float().cpu().numpy()
                else:
                    p = np.zeros(4)
                th = float(np.degrees(np.arctan2(p[1], p[0])))
                vis_results.setdefault(global_idx, {})[cfg_name] = (d, n, p, th)

        m['n'] += bs

    sample_idx += bs
    if sample_idx % (batch_size * 10) == 0:
        print(f'  {sample_idx}/{len(val_ds)} samples done')

print(f'  {len(val_ds)}/{len(val_ds)} samples done')

In [ ]:
# Aggregate and print metrics
summary = {}
for cfg_name in CONFIGS_3:
    m = metrics[cfg_name]
    n = max(1, m['n'])
    summary[cfg_name] = {
        'depth_mse': m['depth_mse'] / n,
        'normal_mse': m['normal_mse'] / n,
        'normal_mse_norm': m['normal_mse_norm'] / n,
        'depth_mae': float(np.mean(m['depth_mae'])) if m['depth_mae'] else float('nan'),
        'depth_rmse': float(np.mean(m['depth_rmse'])) if m['depth_rmse'] else float('nan'),
        'depth_d1': float(np.mean(m['depth_d1'])) if m['depth_d1'] else float('nan'),
        'normal_ang_mean': m['normal_ang_mean'] / n,
        'normal_ang_median': m['normal_ang_median'] / n,
        'pose_rot_l1': m['pose_rot_l1'] / n,
        'pose_rot_deg': m['pose_rot_deg'] / n,
        'pose_trans_l1': m['pose_trans_l1'] / n,
    }

# Write eval_metrics.txt
metrics_path = os.path.join(EVAL_DIR, 'eval_metrics.txt')
with open(metrics_path, 'w') as f:
    f.write('VisTacFusion Real-Only Evaluation Results\n')
    f.write(f'Checkpoint: {ckpt_info}\n')
    f.write(f'Val samples: {len(val_ds)} (real only)\n\n')
    for cfg_name in CONFIGS_3:
        s = summary[cfg_name]
        f.write(f'[{cfg_name}]\n')
        f.write(f'  Depth MSE:               {s["depth_mse"]:.6f}\n')
        f.write(f'  Depth MAE:               {s["depth_mae"]:.6f}\n')
        f.write(f'  Depth RMSE:              {s["depth_rmse"]:.6f}\n')
        f.write(f'  Depth delta<1.25 (%):    {s["depth_d1"]:.2f}\n')
        f.write(f'  Normal MSE:              {s["normal_mse"]:.6f}\n')
        f.write(f'  Normal MSE (norm):       {s["normal_mse_norm"]:.6f}\n')
        f.write(f'  Normal Ang Mean (deg):   {s["normal_ang_mean"]:.2f}\n')
        f.write(f'  Normal Ang Median (deg): {s["normal_ang_median"]:.2f}\n')
        f.write(f'  Pose Rot L1 (cos,sin):   {s["pose_rot_l1"]:.6f}\n')
        f.write(f'  Pose Rot Error (deg):    {s["pose_rot_deg"]:.2f}\n')
        f.write(f'  Pose Trans L1:           {s["pose_trans_l1"]:.6f}\n')
        f.write('\n')

# Print summary table
print('\n' + '=' * 72)
print(f'{"Metric":30s} {"Both":>12s} {"Tactile":>12s} {"RGB":>12s}')
print('=' * 72)
for key in ['depth_mse', 'depth_mae', 'depth_rmse', 'depth_d1',
            'normal_mse', 'normal_mse_norm', 'normal_ang_mean', 'normal_ang_median',
            'pose_rot_l1', 'pose_rot_deg', 'pose_trans_l1']:
    vals = [summary[c].get(key, float('nan')) for c in CONFIGS_3]
    fmt = '.2f' if 'deg' in key or 'd1' in key else '.6f'
    print(f'  {key:28s} {vals[0]:{fmt}} {vals[1]:{fmt}} {vals[2]:{fmt}}')
print('=' * 72)
print(f'\nMetrics saved -> {metrics_path}')

In [ ]:
# Per-object visualization
print(f'Saving one visualization per object ({len(obj_vis_idx)} objects)...')
for obj_name, idx in sorted(obj_vis_idx.items()):
    sample = val_ds[idx]
    results = vis_results.get(idx, {})
    if not results:
        continue

    gt_depth = sample['depth'][0].numpy()
    gt_normal = sample['normal'].permute(1, 2, 0).numpy()
    gt_pose = sample['pose'].numpy()
    tac_img = (sample['tactile'].permute(1, 2, 0).numpy() * std_np + mean_np).clip(0, 255).astype(np.uint8)
    rgb_img = (sample['rgb'].permute(1, 2, 0).numpy() * std_np + mean_np).clip(0, 255).astype(np.uint8)

    save_path = os.path.join(vis_dir, f'{obj_name}.png')
    visualize_three_configs(tac_img, rgb_img, results,
                            gt_depth=gt_depth, gt_normal=gt_normal,
                            gt_pose=gt_pose, save_path=save_path)
print(f'Visualizations saved -> {vis_dir}/')

## 6. Qualitative Visualization (All Real Data)

In [ ]:
from vistacfusion.engine.inference import run_real_data_tree

real_vis_dir = os.path.join(EVAL_DIR, 'real_vis')
os.makedirs(real_vis_dir, exist_ok=True)

print(f'Running real data inference from {LOCAL_REAL}...')
run_real_data_tree(depth_model, LOCAL_REAL, cfg, device, real_vis_dir,
                   max_samples_per_sensor=5, pose_model=pose_model,
                   rgb_subdir='rgb')
print(f'Visualizations saved -> {real_vis_dir}/')

In [ ]:
# Display sample visualizations
from IPython.display import Image as IPImage, display

for vis_dir_name in ['real_val_vis', 'real_vis']:
    vd = os.path.join(EVAL_DIR, vis_dir_name)
    if not os.path.exists(vd):
        continue
    pngs = sorted(glob.glob(os.path.join(vd, '**/*.png'), recursive=True))[:10]
    if pngs:
        print(f'\n=== {vis_dir_name} ({len(pngs)} shown) ===')
        for png in pngs:
            print(os.path.basename(png))
            display(IPImage(filename=png, width=900))